In [ ]:
import numpy as np

GRID_SIZE = 4
TERMINAL_STATES = [(0, 0), (3, 3)]

In [ ]:
ACTIONS = ['up', 'down', 'left', 'right']

def next_state(state, action):
    row, col = state
    if action == 'up':
        row = max(row - 1, 0)
    elif action == 'down':
        row = min(row + 1, GRID_SIZE - 1)
    elif action == 'left':
        col = max(col - 1, 0)
    elif action == 'right':
        col = min(col + 1, GRID_SIZE - 1)
    return (row, col)

In [ ]:
print(next_state((0, 0), 'right'))   # should print (0, 1)
print(next_state((0, 0), 'up'))      # should print (0, 0) - can't leave the grid

(0, 1)
(0, 0)


In [ ]:
def get_reward(state):
    if state in TERMINAL_STATES:
        return 0
    return -1

In [ ]:
def policy_evaluation(theta=0.001, gamma=1.0):
    V = np.zeros((GRID_SIZE, GRID_SIZE))
    iteration = 0
    history = []  # tracks (iteration, max_delta) for your Task 2 table

    while True:
        delta = 0
        new_V = V.copy()
        for row in range(GRID_SIZE):
            for col in range(GRID_SIZE):
                state = (row, col)
                if state in TERMINAL_STATES:
                    continue
                value = 0
                for action in ACTIONS:
                    ns = next_state(state, action)
                    value += 0.25 * (get_reward(state) + gamma * V[ns])
                new_V[row, col] = value
                delta = max(delta, abs(new_V[row, col] - V[row, col]))
        V = new_V
        iteration += 1
        history.append((iteration, delta))
        print(f"Iteration {iteration}: max change = {delta:.4f}")
        if delta < theta:
            break
    return V, history

V_eval, eval_history = policy_evaluation()
print("\nFinal state values (random policy):")
print(V_eval)

Iteration 1: max change = 1.0000
Iteration 2: max change = 1.0000
Iteration 3: max change = 1.0000
Iteration 4: max change = 0.9688
Iteration 5: max change = 0.9375
Iteration 6: max change = 0.8945
Iteration 7: max change = 0.8545
Iteration 8: max change = 0.8110
Iteration 9: max change = 0.7708
Iteration 10: max change = 0.7303
Iteration 11: max change = 0.6925
Iteration 12: max change = 0.6558
Iteration 13: max change = 0.6213
Iteration 14: max change = 0.5883
Iteration 15: max change = 0.5571
Iteration 16: max change = 0.5275
Iteration 17: max change = 0.4995
Iteration 18: max change = 0.4729
Iteration 19: max change = 0.4478
Iteration 20: max change = 0.4240
Iteration 21: max change = 0.4014
Iteration 22: max change = 0.3801
Iteration 23: max change = 0.3598
Iteration 24: max change = 0.3407
Iteration 25: max change = 0.3226
Iteration 26: max change = 0.3054
Iteration 27: max change = 0.2892
Iteration 28: max change = 0.2738
Iteration 29: max change = 0.2592
Iteration 30: max chang

In [ ]:
def value_iteration(theta=0.001, gamma=1.0):
    V = np.zeros((GRID_SIZE, GRID_SIZE))
    policy = {}

    while True:
        delta = 0
        for row in range(GRID_SIZE):
            for col in range(GRID_SIZE):
                state = (row, col)
                if state in TERMINAL_STATES:
                    continue
                action_values = []
                for action in ACTIONS:
                    ns = next_state(state, action)
                    action_values.append(get_reward(state) + gamma * V[ns])
                best_value = max(action_values)
                delta = max(delta, abs(best_value - V[row, col]))
                V[row, col] = best_value
                policy[state] = ACTIONS[np.argmax(action_values)]
        if delta < theta:
            break
    return V, policy

V_opt, optimal_policy = value_iteration()
print("Optimal state values:")
print(V_opt)
print("\nOptimal policy (best action per cell):")
for state, action in optimal_policy.items():
    print(state, "->", action)

Optimal state values:
[[ 0. -1. -2. -3.]
 [-1. -2. -3. -2.]
 [-2. -3. -2. -1.]
 [-3. -2. -1.  0.]]

Optimal policy (best action per cell):
(0, 1) -> left
(0, 2) -> left
(0, 3) -> down
(1, 0) -> up
(1, 1) -> up
(1, 2) -> up
(1, 3) -> down
(2, 0) -> up
(2, 1) -> up
(2, 2) -> down
(2, 3) -> down
(3, 0) -> up
(3, 1) -> right
(3, 2) -> right


In [ ]:
def simulate_path(start, policy_dict=None, random_policy=False, max_steps=50):
    state = start
    path = [state]
    total_reward = 0
    for _ in range(max_steps):
        if state in TERMINAL_STATES:
            return path, total_reward, True
        action = np.random.choice(ACTIONS) if random_policy else policy_dict[state]
        total_reward += get_reward(state)
        state = next_state(state, action)
        path.append(state)
    return path, total_reward, state in TERMINAL_STATES

start = (1, 1)  # pick any non-terminal starting cell

random_path, random_reward, random_reached = simulate_path(start, random_policy=True)
optimal_path, optimal_reward, optimal_reached = simulate_path(start, policy_dict=optimal_policy)

print("Random policy:", len(random_path)-1, "steps, reward:", random_reward, "reached goal:", random_reached)
print("Optimal policy:", len(optimal_path)-1, "steps, reward:", optimal_reward, "reached goal:", optimal_reached)

Random policy: 6 steps, reward: -6 reached goal: True
Optimal policy: 2 steps, reward: -2 reached goal: True


In [ ]:
def greedy_from_values(V):
    policy = {}
    for row in range(GRID_SIZE):
        for col in range(GRID_SIZE):
            state = (row, col)
            if state in TERMINAL_STATES:
                continue
            action_values = []
            for action in ACTIONS:
                ns = next_state(state, action)
                action_values.append(get_reward(state) + V[ns])
            policy[state] = ACTIONS[np.argmax(action_values)]
    return policy

evaluated_policy = greedy_from_values(V_eval)
eval_path, eval_reward, eval_reached = simulate_path(start, policy_dict=evaluated_policy)
print("Evaluated policy:", len(eval_path)-1, "steps, reward:", eval_reward, "reached goal:", eval_reached)

Evaluated policy: 2 steps, reward: -2 reached goal: True
